# Partie 3 — Clustering : combien de segments ?

**Ce qu'on fait ici** : on donne à K-means nos 5 852 clients décrits par R, F et M
(transformés en Partie 2), et on lui demande de les regrouper.

**Le seul vrai problème** : K-means ne devine pas le nombre de groupes. Il faut le lui
donner. Tout ce notebook sert à choisir ce nombre — appelé **k** — et à pouvoir le justifier.

**Plan**
1. Rappel de comment marche K-means
2. On essaie tous les k de 2 à 10 et on mesure la qualité de chacun
3. Trois critères pour choisir : le coude, la silhouette, la stabilité
4. On tranche, on regarde à quoi ressemblent les segments
5. On sauvegarde

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

sns.set_theme(style="whitegrid")
GRAINE = 42                      # fixée pour que les résultats soient reproductibles

PROC = Path("../data/processed")
FIG  = Path("../figures"); FIG.mkdir(parents=True, exist_ok=True)

# rfm      : les valeurs lisibles (jours, nombre de commandes, livres sterling)
# scaled   : les mêmes données transformées (log + standardisation) → entrée de K-means
# scaler   : sert à refaire le chemin inverse pour interpréter les résultats
rfm    = pd.read_parquet(PROC / "rfm_features.parquet")
scaled = pd.read_parquet(PROC / "rfm_scaled.parquet")
meta   = joblib.load(PROC / "scaler.joblib")
scaler, RFM3 = meta["scaler"], meta["cols"]

X = scaled[["R_scaled", "F_scaled", "M_scaled"]].values

print(f"{X.shape[0]:,} clients, {X.shape[1]} variables (R, F, M)")
print("Rappel : ces valeurs sont standardisées, elles tournent autour de 0")
print(np.round(X[:3], 2))

## 1. Comment marche K-means

En trois phrases :

1. On lui dit « fais-moi **k** groupes ». Il place k points au hasard dans l'espace des
   données — ce sont les **centroïdes**, les « centres » provisoires des groupes.
2. Chaque client est affecté au centroïde le plus proche de lui.
3. Chaque centroïde se déplace au milieu des clients qu'il vient de récupérer. On répète
   2 et 3 jusqu'à ce que plus rien ne bouge.

« Le plus proche » = distance euclidienne. C'est pour cela que la Partie 2 était
indispensable : sans mise à l'échelle, le Montant (des milliers) écraserait complètement
la Récence (des centaines) dans ce calcul de distance.

**Deux réglages à connaître** :
- `random_state=42` : fige le hasard de départ, donc le résultat est identique à chaque
  exécution. Sans ça, impossible de reproduire nos chiffres.
- `n_init=20` : relance l'algorithme 20 fois avec des départs différents et garde le
  meilleur résultat. K-means peut tomber sur une solution médiocre selon son point de
  départ ; 20 essais réduisent ce risque.

Essayons une fois, avec k=4, juste pour voir ce que ça produit :

In [ ]:
essai = KMeans(n_clusters=4, n_init=20, random_state=GRAINE).fit(X)

print("Numéro de groupe attribué aux 10 premiers clients :")
print(essai.labels_[:10])
print()
print("Taille de chaque groupe :")
print(pd.Series(essai.labels_).value_counts().sort_index())
print()
print("Inertie :", round(essai.inertia_, 1))
print("(l'inertie = somme des distances² entre chaque client et le centre de son groupe)")
print("(plus elle est basse, plus les groupes sont compacts)")

## 2. On essaie tous les k de 2 à 10

Pour chaque valeur de k, on retient trois choses :

| Mesure | Ce que ça veut dire |
|---|---|
| **inertie** | à quel point les groupes sont compacts — plus bas = plus compact |
| **silhouette** | à quel point les groupes sont bien séparés, entre −1 et 1 — plus haut = mieux |
| **taille du plus petit groupe** | un segment de 50 clients ne sert à rien en marketing |

La troisième colonne est souvent oubliée, alors qu'elle élimine des k à elle seule.

In [ ]:
resultats = []

for k in range(2, 11):
    modele = KMeans(n_clusters=k, n_init=20, random_state=GRAINE).fit(X)
    tailles = np.bincount(modele.labels_)          # nombre de clients par groupe

    resultats.append({
        "k": k,
        "inertie": modele.inertia_,
        # sample_size=5000 : la silhouette est lente à calculer, on l'estime sur un
        # échantillon de 5 000 clients au lieu des 5 852 — le résultat est équivalent
        "silhouette": silhouette_score(X, modele.labels_,
                                       sample_size=5000, random_state=GRAINE),
        "plus_petit_groupe": tailles.min(),
    })

bal = pd.DataFrame(resultats)

# gain_inertie : de combien l'inertie a baissé en passant de k-1 à k.
# C'est ce gain qui sert à repérer le "coude" : il s'écroule quand ajouter un groupe
# n'apporte plus grand-chose.
bal["gain_inertie"] = -bal.inertie.diff()

bal.round(2)

## 3. Trois critères pour choisir k

### Critère 1 — le coude

L'inertie baisse **toujours** quand k augmente (avec un groupe par client, elle vaudrait 0).
On ne cherche donc pas son minimum, mais l'endroit où la courbe **cesse de descendre vite** :
au-delà, chaque groupe supplémentaire n'apporte plus grand-chose.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(bal.k, bal.inertie, "o-")
ax[0].set_title("Inertie — on cherche le coude")
ax[0].set_xlabel("k"); ax[0].set_ylabel("inertie")

ax[1].bar(bal.k[1:], bal.gain_inertie[1:])
ax[1].set_title("Gain obtenu en ajoutant un groupe")
ax[1].set_xlabel("k"); ax[1].set_ylabel("baisse de l'inertie")

plt.tight_layout(); plt.savefig(FIG / "03_coude.png", dpi=150); plt.show()

print("Gain d'inertie apporté par chaque groupe supplémentaire :")
for _, r in bal[1:].iterrows():
    print(f"  passer à k={int(r.k)} fait gagner {r.gain_inertie:,.0f}")

**Ce qu'on lit** : le gain passe de 2 224 (k=3) à 1 430 (k=4) puis 818 (k=5). Ça fléchit
progressivement entre 3 et 5, **sans angle net**.

C'est la limite bien connue de cette méthode : elle repose sur une lecture visuelle.
Trois personnes peuvent voir le coude à trois endroits différents. Elle nous donne une
**zone** (k entre 3 et 5), pas une réponse. Il faut d'autres critères pour trancher dedans.

### Critère 2 — la silhouette

Pour chaque client, la silhouette compare deux distances : celle qui le sépare des membres
de son propre groupe, et celle qui le sépare du groupe voisin le plus proche.
Proche des siens et loin des autres → score proche de 1. À cheval entre deux groupes →
score proche de 0. Mal classé → score négatif.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bal.k, bal.silhouette, "o-", color="#55A868")
ax.set_xlabel("k"); ax.set_ylabel("silhouette")
ax.set_title("Silhouette selon k")
plt.tight_layout(); plt.savefig(FIG / "03_silhouette.png", dpi=150); plt.show()

for _, r in bal.iterrows():
    print(f"  k={int(r.k):2d} : silhouette {r.silhouette:.3f} | "
          f"plus petit groupe {int(r.plus_petit_groupe):,} clients")

**⚠️ Le piège — c'est la question 4 du rapport**

La silhouette est **maximale à k=2** (0,437). Si on suivait la métrique, on livrerait deux
segments : « clients actifs » et « clients partis ».

Techniquement correct. Commercialement inutile — on ne construit aucune campagne
différenciée avec deux groupes aussi grossiers.

Pourquoi elle se trompe ici :
- elle mesure une **géométrie** (des groupes compacts et éloignés), pas une utilité ;
- elle **favorise mécaniquement les petits k** : moins il y a de groupes, plus ils sont
  éloignés les uns des autres ;
- elle **ignore la taille** des groupes.

Ce qui reste utile, ce n'est pas sa valeur absolue mais ses **variations** : elle chute de
k=2 à k=3, puis **remonte à k=4**. Ce petit rebond est un signal.

### Critère 3 — la stabilité (le plus solide)

L'idée : si nos segments sont réels, ils doivent **réapparaître** quand on change un peu
les données. On tire au hasard 80 % des clients, on refait tourner K-means dessus, et on
regarde si les clients sont regroupés de la même façon qu'avant. On recommence 20 fois.

La comparaison se fait avec l'**ARI** : 1 = les deux découpages sont identiques,
0 = ils ne se ressemblent pas plus que par hasard.

C'est le critère le plus convaincant, parce qu'il ne mesure pas une forme géométrique mais
une **reproductibilité** — exactement ce qu'on nous demande de prouver en question 5.

In [ ]:
def tester_stabilite(X, k, n_essais=20):
    """Compare le découpage de référence à 20 découpages obtenus sur 80 % des clients."""
    tirage = np.random.default_rng(GRAINE)

    # découpage de référence, sur la totalité des clients
    reference = KMeans(k, n_init=20, random_state=GRAINE).fit(X)

    scores = []
    for essai in range(n_essais):
        # on tire 80 % des clients au hasard, sans remise
        echantillon = tirage.choice(len(X), int(0.8 * len(X)), replace=False)

        # on reclusterise uniquement sur ces clients-là
        modele = KMeans(k, n_init=10, random_state=1000 + essai).fit(X[echantillon])

        # on compare : ces clients sont-ils regroupés comme dans la référence ?
        scores.append(adjusted_rand_score(reference.labels_[echantillon], modele.labels_))

    return np.array(scores)


stabilite = {}
for k in range(3, 7):
    scores = tester_stabilite(X, k)
    stabilite[k] = scores
    print(f"k={k} : ARI moyen {scores.mean():.3f} | le pire des 20 essais : {scores.min():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=pd.DataFrame(stabilite), ax=ax)
ax.axhline(0.9, ls="--", c="crimson", alpha=.6)
ax.set_xlabel("k"); ax.set_ylabel("ARI (1 = découpage identique)")
ax.set_title("Stabilité : retrouve-t-on les mêmes segments sur 80 % des clients ?")
plt.tight_layout(); plt.savefig(FIG / "03_stabilite.png", dpi=150); plt.show()

**Comment lire** : au-dessus de 0,90 la partition est solide. Entre 0,75 et 0,90 elle est
acceptable. En dessous de 0,75, les segments dépendent trop de l'échantillon pour qu'on
puisse s'appuyer dessus.

k=4 obtient le meilleur score (0,961) et k=6 se dégrade nettement — à partir de six groupes,
les frontières bougent d'un tirage à l'autre.

## 4. La décision

Aucun critère ne suffit seul. On les met côte à côte :

In [ ]:
recap = bal[bal.k.between(2, 6)][["k", "silhouette", "plus_petit_groupe", "gain_inertie"]].copy()
recap["ARI_stabilite"] = recap.k.map({k: v.mean() for k, v in stabilite.items()})
recap.round(3)

| k | Verdict |
|---|---|
| 2 | Meilleure silhouette, mais deux segments n'ont aucune utilité marketing |
| 3 | Correct partout, mais perd la distinction entre nouveaux clients et clients perdus |
| **4** | **Rebond de silhouette, meilleure stabilité, tous les groupes > 1 100 clients** |
| 5 | Un groupe tombe à 450 clients, stabilité en baisse |
| 6 | Stabilité nettement dégradée (0,901, avec un essai à 0,789) |

**On retient k = 4.** Non parce qu'un chiffre le prouve, mais parce que quatre indications
indépendantes pointent au même endroit — et parce que les quatre groupes obtenus se
racontent chacun en une phrase (section suivante).

C'est la réponse à la question 5 du rapport : sans vérité terrain, la défense repose sur la
**convergence de plusieurs critères** plus la **stabilité**, pas sur une métrique unique.

In [ ]:
K = 4

kmeans = KMeans(n_clusters=K, n_init=20, random_state=GRAINE).fit(X)
rfm["cluster"] = kmeans.labels_

print(f"k retenu : {K}")
print(rfm.cluster.value_counts().sort_index())

## 5. À quoi ressemblent ces 4 segments

Les centroïdes calculés par K-means sont exprimés en valeurs standardisées — illisibles.
On refait le chemin inverse de la Partie 2 pour retrouver des jours et des livres :

`inverse_transform` annule la standardisation, puis `expm1` annule le `log1p`.

In [ ]:
centres = np.expm1(scaler.inverse_transform(kmeans.cluster_centers_))

profils = pd.DataFrame(centres, columns=RFM3).round(0)
profils["effectif"] = np.bincount(kmeans.labels_)
profils["part_CA_%"] = (100 * rfm.groupby("cluster").Montant.sum()
                        / rfm.Montant.sum()).round(1).values

profils.sort_values("Montant", ascending=False)

Chaque ligne se lit directement : « ce groupe a acheté il y a X jours, Y fois, pour Z £ ».
C'est exactement pour garder cette lisibilité qu'on a refusé la PCA en Partie 2.

Le nommage de ces groupes se fait en Partie 4.

### Vue d'ensemble
On projette les clients en 2 dimensions pour visualiser le découpage.
(La Partie 2 a montré que 2 axes suffisent à représenter 95 % de l'information.)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=GRAINE)
plan = pca.fit_transform(X)
centres_projetes = pca.transform(kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(plan[:, 0], plan[:, 1], c=kmeans.labels_, s=6, alpha=.45, cmap="viridis")
ax.scatter(centres_projetes[:, 0], centres_projetes[:, 1],
           c="red", s=250, marker="X", edgecolor="white", linewidth=1.5)
ax.set_xlabel(f"axe 1 ({pca.explained_variance_ratio_[0]:.0%}) — valeur du client")
ax.set_ylabel(f"axe 2 ({pca.explained_variance_ratio_[1]:.0%}) — récence")
ax.set_title(f"Les {K} segments (croix rouges = centres)")
plt.tight_layout(); plt.savefig(FIG / "03_projection.png", dpi=150); plt.show()

**À remarquer** : les groupes sont bien distincts, mais il n'y a **pas de vide entre eux**.
Les clients forment un continuum, pas des grappes naturellement isolées.

Autrement dit : nous ne « découvrons » pas des groupes qui existaient déjà, nous découpons
un espace continu à des endroits utiles. C'est une nuance honnête à dire soi-même à l'oral,
et elle explique aussi pourquoi la silhouette reste modeste (0,365) — c'est normal sur des
données comportementales.

## 6. Les retours changent-ils quelque chose ? (question 1 du rapport)

En Partie 1, on avait mis les annulations de côté et préparé un second jeu où elles sont
déduites du CA. On refait tourner K-means dessus et on compare les deux découpages par ARI.

Réponse chiffrée, pas une opinion.

In [ ]:
scaled_net = pd.read_parquet(PROC / "rfm_scaled_net.parquet")
X_net = scaled_net[["R_scaled", "F_scaled", "M_scaled"]].values

kmeans_net = KMeans(K, n_init=20, random_state=GRAINE).fit(X_net)

# on rapproche les deux découpages client par client
comparaison = (
    pd.DataFrame({"CustomerID": scaled.CustomerID,     "sans_retours": kmeans.labels_})
    .merge(
    pd.DataFrame({"CustomerID": scaled_net.CustomerID, "avec_retours": kmeans_net.labels_}),
    on="CustomerID")
)

ari = adjusted_rand_score(comparaison.sans_retours, comparaison.avec_retours)

print(f"Clients présents dans les deux jeux : {len(comparaison):,}")
print(f"ARI entre les deux segmentations   : {ari:.3f}")
print()
print("Combien de clients changent de groupe :")
print(pd.crosstab(comparaison.sans_retours, comparaison.avec_retours))

**Comment interpréter l'ARI obtenu**

| Valeur | Conclusion à écrire dans le rapport |
|---|---|
| > 0,85 | Le traitement des retours ne change pas la structure. On garde le jeu le plus simple. |
| 0,60 – 0,85 | Effet réel sur les frontières. Regarder le tableau croisé : quels segments bougent ? |
| < 0,60 | Choix structurant. Il faut trancher explicitement et justifier. |

## 7. Sauvegarde

In [ ]:
rfm.to_parquet(PROC / "rfm_clusters.parquet", index=False)
joblib.dump({"kmeans": kmeans, "k": K, "random_state": GRAINE},
            PROC / "modele_kmeans.joblib")
bal.to_csv(PROC / "balayage_k.csv", index=False)

print("Sauvegardé :")
print("  rfm_clusters.parquet  → les clients avec leur numéro de segment (Partie 4)")
print("  modele_kmeans.joblib  → le modèle, pour classer de nouveaux clients")
print("  balayage_k.csv        → le tableau des métriques, pour le rapport")

## Résumé pour le rapport

> Le clustering retenu est un **K-means à k = 4** (`n_init=20`, `random_state=42`) appliqué
> aux variables RFM transformées en log puis standardisées. Le choix de k ne repose pas sur
> une métrique unique : la silhouette est maximale à k=2 (0,437), un optimum techniquement
> correct mais commercialement inexploitable, et le coude de l'inertie fléchit
> progressivement entre k=3 et k=5 sans angle net. Nous retenons k=4 comme point de
> convergence de quatre indications — rebond de la silhouette (0,365), gain d'inertie encore
> substantiel, plus petit segment supérieur à 1 100 clients, et surtout **stabilité par
> bootstrap : ARI moyen de 0,961 sur 20 rééchantillonnages à 80 % des clients**. Les quatre
> segments obtenus sont nettement contrastés : 1 184 clients très actifs concentrent 73 % du
> chiffre d'affaires, tandis que 1 968 clients mono-achat anciens n'en pèsent que 3,7 %.
> La projection en deux dimensions montre des frontières franches mais un espace continu :
> le clustering découpe un continuum de comportements à des seuils utiles, il ne révèle pas
> des groupes naturels préexistants.

**Questions traitées** : Q1 (effet des retours) · Q4 (limites de la silhouette) ·
Q5 (validation sans vérité terrain)